In [1]:
import gzip
import json
import os
import shutil
import sys
from collections.abc import Iterable, Iterator
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from dotenv import load_dotenv

In [2]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "archive").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache

sys.path.insert(0, str(REPO))  # predict.py lives at the repo root
load_dotenv(REPO / ".env")

True

In [3]:
def _resolve_paths(source: str | Path | Iterable[str | Path]) -> list[Path]:
    if isinstance(source, (str, Path)):
        p = Path(source)
        if p.is_dir():
            return sorted(p.glob("*.jsonl.gz"))
        return [p]
    return [Path(p) for p in source]

def _parse_filename(path: Path) -> tuple[str, str]:
    """``EARNINGS_RELEASE_2025Q3.jsonl.gz`` → ``("EARNINGS_RELEASE", "2025Q3")``."""
    stem = path.name.removesuffix(".jsonl.gz")
    event_type, _, quarter = stem.rpartition("_")
    return event_type, quarter

def read_jsonl_gz(path: str | Path) -> Iterator[dict]:
    """Yield each record from a gzip-JSONL file, skipping blank lines."""
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)

def _load_one(path: Path) -> pd.DataFrame:
    df = pd.DataFrame(read_jsonl_gz(path))
    event_type, quarter = _parse_filename(path)
    # Add provenance from the filename, but never clobber a field the record
    # already carries. Insert at the front so it reads left-to-right.
    if "quarter" not in df.columns:
        df.insert(0, "quarter", quarter)
    if "event_type" not in df.columns:
        df.insert(0, "event_type", event_type)
    return df

def load_archive(source: str | Path | Iterable[str | Path]) -> pd.DataFrame:
    """Load one or more ``*.jsonl.gz`` archive files into a single DataFrame.

    ``source`` may be a directory (all ``*.jsonl.gz`` within it), a single file,
    or an iterable of paths. Provenance columns ``event_type`` and ``quarter``
    are added from each filename — unless a record already carries that field, in
    which case the record's own value is kept.
    """
    frames = [_load_one(p) for p in _resolve_paths(source)]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [4]:
df_archive = load_archive(ARCHIVE_DIR)

## Paid dry run of `predict()` on two archived events

Runs the deployed entry point end to end — `predict.py` → `em_agent` graph → shadow records — on two scored **2026Q4** events (after the pooled model's training data, all 2026Q3 preview events):

| Case | Branch | LLM calls |
|---|---|---|
| with the competition's preview | extract → classify → features → pooled ridge | gpt-6-luna (medium) extraction + gpt-6-luna (low) classification |
| without any preview | direct prediction | one gpt-6.1-sol (medium) call |

**This spends real OpenAI credit** (a few cents per event). Nothing is submitted to the competition. The bundle comes from the archive instead of `information_url`, and records go to a local folder `data/surprise/dry_run/` (`SHADOW_STORE_DIR`), laid out as on the Modal Volume. If the `LANGFUSE_*` keys are in `.env`, the runs are traced.

To exercise the *own preview* branch instead, save a preview for the no-preview event before running predict: `store.save_own_preview(event_id, {"preview": text, "usable": True})`.

In [5]:
QUARTER = "2026Q4"
WEBHOOK_FIELDS = ["event_id", "event_type", "timing_category", "event_datetime", "knowledge_cutoff", "focal_assets"]


def has_preview(disclosure) -> bool:
    items = disclosure.get("items") or [] if isinstance(disclosure, dict) else []
    return any(isinstance(i, dict) and i.get("id") == "earnings-preview" and str(i.get("content") or "").strip() for i in items)


def car1(row) -> float | None:
    returns = row["event_returns"] if isinstance(row["event_returns"], dict) else {}
    ret = returns.get(row["focal_assets"][0]["identifier_value"]) or {}
    return ret.get("car1") if ret.get("return_status") == "ok" else None


quarter = df_archive[(df_archive["quarter"] == QUARTER) & (df_archive["status"] == "scored")].copy()
quarter["ticker"] = quarter["focal_assets"].map(lambda a: a[0]["identifier_value"])
quarter["has_preview"] = quarter["disclosure"].map(has_preview)
quarter["car1"] = quarter.apply(car1, axis=1).astype(float)
# The target: rank of car1 within the quarter (only a rough reference with this few scored events).
quarter["actual_pct"] = (quarter["car1"].rank() - 1) / (quarter["car1"].notna().sum() - 1)

picks = {
    "competition preview": quarter[quarter["has_preview"]].sort_values("event_datetime").iloc[-1],
    "no preview": quarter[~quarter["has_preview"]].sort_values("event_datetime").iloc[-1],
}
pd.DataFrame(picks).T[["event_id", "ticker", "event_datetime", "has_preview", "car1", "actual_pct"]]

,event_id,ticker,event_datetime,has_preview,car1,actual_pct
competition preview,ea_RELL_Q1_2027,RELL,2026-10-08T14:00:00+00:00,True,0.002358,0.45
no preview,ea_SAR_Q2_2027,SAR,2026-10-07T14:00:00+00:00,False,-0.052979,0.1


In [6]:
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing: predict() would only return the 0.5 fallback"

DRY_RUN_DIR = REPO / "data" / "surprise" / "dry_run"
shutil.rmtree(DRY_RUN_DIR, ignore_errors=True)  # start clean, so no earlier own preview is picked up
os.environ["SHADOW_STORE_DIR"] = str(DRY_RUN_DIR)
os.environ.pop("SHADOW_VOLUME", None)

import predict as predict_module
from shadow import store as shadow_store, sync

# predict() fetches the bundle with httpx.get(information_url); serve the archived bundle instead.
BUNDLES = {}
predict_module.httpx = SimpleNamespace(get=lambda url, **kwargs: SimpleNamespace(
    raise_for_status=lambda: None, json=lambda: BUNDLES[url]))
store = shadow_store.EventStore(DRY_RUN_DIR)

In [7]:
# PAID: one extraction + one classification call, and one direct-prediction call.
results = []
for case, row in picks.items():
    event = {k: row[k] for k in WEBHOOK_FIELDS} | {"information_url": f"archive://{row['event_id']}"}
    BUNDLES[event["information_url"]] = row["disclosure"]
    predictions = predict_module.predict(event)
    record = store.read_json(event["event_id"], shadow_store.PREDICTION)
    results.append({"case": case, "event_id": event["event_id"], "preview_source": record["preview_source"],
                    "branch": record["branch"], "prediction": predictions[0]["predicted_percentile"],
                    "actual_pct": row["actual_pct"], "error": record["error"]})
pd.DataFrame(results)

[WARN] Langfuse host http://localhost:3000 is unreachable; tracing is off.


,case,event_id,preview_source,branch,prediction,actual_pct,error
0,competition preview,ea_RELL_Q1_2027,competition,surprise_ridge,0.697322,0.45,None
1,no preview,ea_SAR_Q2_2027,NaN,direct,0.240000,0.10,None


### What was recorded

Each event's folder has `competition_preview.md` (if the bundle had one), `surprises.json`, `features.json` (surprise branch only) and `prediction.json`. `shadow.sync` loads them the same way it loads files downloaded from Modal.

In [8]:
for case, row in picks.items():
    folder = store.path(row["event_id"], shadow_store.PREDICTION).parent
    print(f"{case}: {sorted(p.name for p in folder.iterdir())}")
    reasons = store.read_json(row["event_id"], shadow_store.PREDICTION)["reasons"]
    if reasons:
        print(f"  reasons: {reasons}")

surprises = store.read_json(picks["competition preview"]["event_id"], shadow_store.SURPRISES)
pd.DataFrame(surprises["items"])[["position", "dimension", "direction", "magnitude", "description"]]

competition preview: ['competition_preview.md', 'features.json', 'prediction.json', 'surprises.json']
no preview: ['prediction.json']
  reasons: 1. NAV per share fell 4.6% sequentially, with $0.82 of credit-related markdowns indicating meaningful asset-quality deterioration despite buyback accretion.
2. Adjusted NII of $0.46 covers only about 61% of the $0.75 quarterly base dividend. With spillover reduced to roughly $1 per share, distribution sustainability is a material concern.
3. Earnings pressure appears persistent: refinancing increased interest expense, while replacement originations carried spreads 220 basis points below the loans repaid.
4. Important offsets limit the downside forecast: nonaccruals reached zero after asset sales, the CLO reset restores fees and interest income, and refinancing reduced near-term maturity risk.
5. The shares already trade roughly 25% below reported NAV, suggesting some distress may be priced in. With no stated consensus comparison or option-mark

,position,dimension,direction,magnitude,description
0,0,revenue,positive,large,"Sales were $64.9 million, about 10% above the ..."
1,1,bottom_line_earnings,positive,large,Diluted EPS of $0.27 far exceeded both the $0....
2,2,operating_profitability,positive,large,"Gross margin reached 34.6%, well above the pre..."
3,3,operating_kpis,positive,large,"Backlog rose to $184.4 million, up $20 million..."
4,4,revenue,positive,moderate,PMT sales were $46.6 million excluding legacy ...
5,5,forward_guidance,positive,moderate,"GES sales rose 27.1%, about $2 million, and ma..."
6,6,cash_balance_sheet_liquidity,positive,moderate,Operating cash flow was $5.9 million and cash ...


In [9]:
# Nonzero features of the surprise-branch event, with each one's contribution to the score
# (ridge weight x the feature after the pipeline's imputation and scaling).
from em_agent.nodes import pooled_model

fitted = pooled_model()
row = pd.DataFrame([store.read_json(picks["competition preview"]["event_id"], shadow_store.FEATURES)["features"]])
prep, ridge = fitted.pipeline.named_steps["prep"], fitted.pipeline.named_steps["ridge"]
terms = pd.DataFrame({"feature": prep.get_feature_names_out(), "transformed": prep.transform(row[fitted.features])[0],
                      "weight": ridge.coef_})
terms["value"] = terms["feature"].map(lambda f: row[f].iloc[0] if f in row else None)
terms["contribution"] = terms["transformed"] * terms["weight"]
print(f"intercept {ridge.intercept_:.4f} + contributions {terms['contribution'].sum():.4f} = prediction (before clipping)")
terms[terms["value"].fillna(1) != 0].sort_values("contribution", key=abs, ascending=False).round(4)

intercept 0.4999 + contributions 0.1975 = prediction (before clipping)


,feature,transformed,weight,value,contribution
52,net_w_first3,1.4773,0.0159,9.0000,0.0235
50,net_w,1.6775,0.0117,18.0000,0.0196
53,net_per_item,1.5856,0.0116,1.0000,0.0184
16,pos_w__revenue,1.8871,0.0087,5.0000,0.0165
1,net_w__revenue,1.7048,0.0091,5.0000,0.0155
63,opt_skew_25_delta,2.7934,0.0055,0.1454,0.0153
51,net_first3,1.0477,0.0134,3.0000,0.0140
49,net,1.2578,0.0095,7.0000,0.0119
18,pos_w__operating_profitability,1.1668,0.0073,3.0000,0.0086
3,net_w__operating_profitability,1.1408,0.0074,3.0000,0.0085


In [10]:
display(sync.load_predictions(DRY_RUN_DIR))
sync.load_features(DRY_RUN_DIR)

,event_id,event_datetime,knowledge_cutoff,preview_source,branch,reasons,error,model,created_at,identifier_value,predicted_percentile
0,ea_RELL_Q1_2027,2026-10-08T14:00:00+00:00,2026-10-07T20:00:00+00:00,competition,surprise_ridge,NaN,None,em_agent/artifacts/pooled_ridge_v2.joblib,2026-10-26T06:08:57.157359+00:00,RELL,0.697322
1,ea_SAR_Q2_2027,2026-10-07T14:00:00+00:00,2026-10-06T20:00:00+00:00,NaN,direct,"1. NAV per share fell 4.6% sequentially, with ...",None,NaN,2026-10-26T06:09:07.395131+00:00,SAR,0.240000


,event_id,net_w__forward_guidance,net_w__revenue,net_w__operating_kpis,net_w__operating_profitability,net_w__bottom_line_earnings,net_w__cash_balance_sheet_liquidity,net_w__development_execution_regulatory,net_w__commercial_economics,net_w__capital_allocation,...,opt_implied_earnings_volatility,opt_implied_absolute_earnings_move,opt_skew_25_delta,has_option_stats,preview_source,src_own,net_w__x__src_own,net_w_first3__x__src_own,n_items__x__src_own,no_surprises__x__src_own
0,ea_RELL_Q1_2027,2,5,3,3,3,2,0,0,0,...,0.1656,0.132,0.1454,1,competition,0,0,0,0,0
